https://rbrooksdk.github.io/STA1_26/assignments/assignment_03_sampling_and_estimation/

In [130]:
from statsmodels.stats.proportion import proportion_confint
from IPython.display import display, Markdown
import scipy.stats as stats
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import math

# Part 1 — Sampling distributions and the CLT
#### Restoration time in a service system is modelled as exponential with population mean and standard deviation $\mu=\sigma=60$ minutes. Use this known population model and NumPy's random generator with seed 2026 and at least 20,000 repetitions per sample size.

### a) Simulate samples for $n=1,5,20,80$ and store their means. Create a table with , theoretical mean, simulated mean of means, theoretical standard error $\sigma/\sqrt n$, and empirical standard deviation of the simulated means.

### b) Create one figure showing the four sampling distributions with density histograms and overlay $N(\mu,\sigma^2/n)$. Explain how centre, spread, and shape change and why the normal curve is an approximation for this non-normal population.

### c) For $n = 20$ and $n = 80$ , estimate $P(\bar X>75)$ by simulation and calculate the CLT approximation. Compare the values.

# Part 3 — Estimating a false-rejection proportion
#### The file assignment03_access_control.csv contains 240 independently staged legitimate login attempts. false_reject equals 1 when a legitimate attempt was rejected and 0 otherwise. Let $p$ be the population false-rejection probability under the test conditions.

In [131]:
ac = pd.read_csv("assignment03_access_control.csv")
ac.head(5)

,attempt_id,false_reject
0,1,0
1,2,0
2,3,0
3,4,1
4,5,0


### a) Report $n$, the number of false rejections $x$, and $\hat p$. Explain why $\hat p$ is also the sample mean of a binary variable.

In [132]:
n = len(ac)
x = ac['false_reject'].sum()
p_hat = x / n

display(Markdown(f"""
$n = {n}$ \\
$x = {x}$ \\
$\\hat p = {p_hat}$ \\
$\\hat p$ is equal to a sample mean of a binary variable since its values are only zeros or ones, so adding up all values is the same as counting the number of occurances of 1's
# """))


$n = 240$ \
$x = 20$ \
$\hat p = 0.08333333333333333$ \
$\hat p$ is equal to a sample mean of a binary variable since its values are only zeros or ones, so adding up all values is the same as counting the number of occurances of 1's
# 

### b) Construct and interpret a 95% Wilson interval for $p$ with statsmodels. State the population and parameter clearly.

In [ ]:
wilson = proportion_confint(count=x, nobs=n, alpha=0.05, method='wilson')
display(Markdown(f"""
95% wilson = {tuple(np.round(wilson, 4))} \\
The population would be all legitimate login attempts, while the parameter is the true probability of getting a false reject which, due to the wilson interval, we are 95% sure is between 0.055 and 0.125
"""))


95% wilson = (np.float64(0.0546), np.float64(0.1252)) \
The population would be all legitimate login attempts, while the parameter is the true probability of getting a false reject which, due to the wilson interval, we are 95% is between 0.055 and 0.125


### c) Calculate the 95% Wald interval from its formula. Compare it with Wilson and explain why Wilson is the preferred default.

In [134]:
wald_se = np.sqrt(p_hat * (1 - p_hat) / n)
z = stats.norm.ppf(0.975)
wald = (p_hat - z * wald_se, p_hat + z * wald_se)
display(Markdown(f"""
95% wald vs 95% wilson \\
{tuple(np.round(wald, 4))} &emsp; {tuple(np.round(wilson, 4))} \\
Wilson is preferable as wald requires a large sample size
"""))


95% wald vs 95% wilson \
(np.float64(0.0484), np.float64(0.1183)) &emsp; (np.float64(0.0546), np.float64(0.1252)) \
Wilson is preferable as wald requires a large sample size


### d) Create a table of 90%, 95%, and 99% Wilson intervals and widths.

In [135]:
to_print = "|wilson | low | high | width | \n| -------: | ---: | ---: | ---: |\n"
for alp in [0.1, 0.05, 0.01]:
    (w_low, w_high) = proportion_confint(count=x, nobs=n, alpha=alp, method='wilson')
    to_print += f"|{1 - alp} | {round(w_low, 4)} | {round(w_high, 4)}| {round((w_high - w_low) ,4 )} | \n"
display(Markdown(to_print))

|wilson | low | high | width | 
| -------: | ---: | ---: | ---: |
|0.9 | 0.0584 | 0.1175| 0.0591 | 
|0.95 | 0.0546 | 0.1252| 0.0706 | 
|0.99 | 0.0478 | 0.1412| 0.0934 | 


### e) Estimate the sample size for a 95% margin of error no larger than 0.02 using $p = 0.5$ first and then the $\hat p$ observed. Round up and explain the difference.

In [136]:
def stupid_sample_search(p):
    for x in range(1, 10000):
        wald_se = np.sqrt(p * (1 - p) / x)
        z_crit = stats.norm.ppf(0.975)
        moe = wald_se * z_crit
        if (moe < 0.02):
            return x

display(Markdown(f"""
The sample size required for a 95% MOE no larger than 0.02 for p=0.5 is {stupid_sample_search(0.5)} \\
While for the actual observed $\\hat p = {p_hat}$ it is {stupid_sample_search(p_hat)} \\
The difference would come from the fact that $0.5 \\neq {p_hat}$
"""))


The sample size required for a 95% MOE no larger than 0.02 for p=0.5 is 2401 \
While for the actual observed $\hat p = 0.08333333333333333$ it is 734 \
The difference would come from the fact that $0.5 \neq 0.08333333333333333$


# Part 4 — AI-use and conclusion